In [1]:
import numpy as np
import math
import json
import time
from IPython.display import HTML
from line_profiler import LineProfiler

from ipynb.fs.full.forcemodel import IForceModel, GravityForce
from ipynb.fs.full.integrator import IIntegrator, SymplecticEuler, VelocityVerlet
from ipynb.fs.full.env import SolarSystemEnv, run_simulation, run_script_simulation, adjust_barycentric
from ipynb.fs.full.vis import visualize_trajectories, plot_static_trajectories
from ipynb.fs.full.helpers import save_states, load_states
from ipynb.fs.full.ship import SimpleImpulseShip, Maneuver, simple_action_space
from ipynb.fs.full.rlmontecarlo import train_mc, MonteCarloAgent, PolicyNet, compute_returns

In [2]:
#%load_ext line_profiler

Cool but unstable with really massive suns
bodies = [
    # Masses (kg)
    np.array([
        1.0e31,
        1.0e31,
        1.0e31,
    ]),
    # Positions (x, y) in meters
    np.array([
        [0.97e11, -0.243e11],   
        [-0.97e11, 0.243e11],
        [0.0, 0.0], 
    ]),
    # Velocities (vx, vy) in m/s
    np.array([
        [0.466e5, 0.432e5],    
        [0.466e5, 0.432e5],
        [-0.932e5, -0.864e5], 
    ]),
    # Names
    np.array([
        "One",
        "Two",
        "Three",
    ]),
    # Colors (reasonable defaults for visualization)
    np.array([
        "red",        
        "blue",
        "green",
    ]),
]

In [3]:
# This precompiles simulation for njit
bodies = [
    # Masses (kg)
    np.array([
        1.5e10,
        1.5e10
    ]),
    # Positions (x, y) in meters (approx semi-major axes on x-axis)
    np.array([
        [0.97, -0.243],
        [-0.97, 0.243]
    ]),
    # Velocities (vx, vy) in m/s (circular orbit approximation)
    np.array([
        [0.466, 0.432],
        [0.466, 0.432]
    ]),
    # Names
    np.array([
        "One",
        "Two"
    ]),
    # Colors (reasonable defaults for visualization)
    np.array([
        "red",
        "blue"
    ]),
]
dt=0.0001
n_steps=1000
states = run_simulation(GravityForce, VelocityVerlet, bodies, dt=dt, n_steps=n_steps, records_len=500)

# Monte Carlo RL with NN-Q Table #

In [4]:
bodies = [
    # Masses
    np.array([
        5.972e24,       # Earth
        7.342e22,       # Luna
        1.0e5,          # Ship
    ]),
    # Positions (x, y) in meters
    np.array([
        [0.0, 0.0],                # Earth
        [3.844e8, 0.0],      # Luna
        [3.6e7, 0.0],        # Ship
    ]),
    # Velocities (vx, vy) in m/s
    np.array([
        [0.0, 0.0],                   # Earth
        [0.0, 1022.0],          # Luna
        [0.0, 3055.0],         # Ship
    ]),
    # Names of celestial bodies
    np.array([
        "Earth",
        "Moon",
        "Ship"
    ]),
    # Colors for visuals
    np.array([
        "blue",       # Earth
        "gray",       # Moon (Luna)
        "lightblue",  # Ship
    ]),
]

In [5]:
bodies = [
    # Masses
    np.array([
        7.342e22,       # Luna
        7.342e22,       # Luna
        1.0e5,          # Ship
    ]),
    # Positions (x, y) in meters
    np.array([
        [3.844e8, 0.0],      # Luna
        [-3.844e8, 0.0],      # Luna
        [0.0, 0.0],        # Ship
    ]),
    # Velocities (vx, vy) in m/s
    np.array([
        [0.0, 0.0],          # Luna
        [0.0, 0.0],          # Luna
        [0.0, 0.0],         # Ship
    ]),
    # Names of celestial bodies
    np.array([
        "Moon",
        "Moon2",
        "Ship"
    ]),
    # Colors for visuals
    np.array([
        "gray",       # Moon (Luna)
        "gray",       # Moon (Luna)
        "lightblue",  # Ship
    ]),
]

ship_index = len(bodies[0])-1
mass = bodies[0][-1]
thrust = 1e4 # thrust changes with dt so keep that in mind
actions = simple_action_space 
safety_radius = 1e4 # on a space scale 10 km is defenitely an atmo entry
escape_dist = 6e8   
escape_vel = 1e5 
target_dist = 1e5 # for a low lunar orbit
target_vel = 1700 # for a low lunar orbit
target_index = 0
pos_scale = 1e7
vel_scale = 1e4

n_actions = actions.n
n_episodes=1000

dt=60.0
simulation_time = 3600 * 24 * 20
max_steps= int(simulation_time/dt)

reward_coef = 1.0 / max_steps

In [9]:
ship = SimpleImpulseShip(ship_index=ship_index, mass=mass,thrust=thrust, actions=actions, safety_radius=safety_radius, escape_dist=escape_dist, escape_vel=escape_vel, target_dist=target_dist, target_vel=target_vel, target_index=target_index, pos_scale=pos_scale, vel_scale=vel_scale, reward_coef=reward_coef)

env = SolarSystemEnv(GravityForce, VelocityVerlet, bodies, ship=ship, dt=dt)

obs0 = env.reset()
obs_dim = obs0.shape[0]

agent = MonteCarloAgent(obs_dim, n_actions)

lp = LineProfiler()
lp.add_function(MonteCarloAgent.run_episode)
lp.add_function(MonteCarloAgent.update_policy)

agent = MonteCarloAgent(obs_dim, n_actions)

lp_wrapper = lp(train_mc)
lp_wrapper(env, agent, n_episodes=10, max_steps=10000, log_every=20, log_states=False, log_n_entries=400)

lp.print_stats()
#states = train_mc(env, agent, n_episodes=n_episodes, max_steps=max_steps, log_every=20, log_states=True, log_n_entries=400)
#%lprun -f MonteCarloAgent.run_episode -f MonteCarloAgent.update_policy train_mc(env, agent, n_episodes=1, max_steps=1000, log_every=20, log_states=False, log_n_entries=400)

Failure: Ship escaped the system or exceeded system escape velocity
Failure: Ship escaped the system or exceeded system escape velocity
Failure: Ship escaped the system or exceeded system escape velocity
Failure: Ship escaped the system or exceeded system escape velocity
Failure: Ship escaped the system or exceeded system escape velocity
Failure: Ship escaped the system or exceeded system escape velocity
Failure: Ship escaped the system or exceeded system escape velocity
Failure: Ship escaped the system or exceeded system escape velocity
Failure: Ship escaped the system or exceeded system escape velocity
Failure: Ship escaped the system or exceeded system escape velocity
Timer unit: 1e-07 s

Total time: 28.4874 s
File: D:\Desktop\jupo\src\rlmontecarlo.ipynb
Function: MonteCarloAgent.run_episode at line 85

Line #      Hits         Time  Per Hit   % Time  Line Contents
    85                                               "        return Categorical(logits=logits)"

Total time: 10.8014 s

In [7]:
scale=1.0/4e8
centre_body="Earth"
xlim=(-1, 1)
ylim=(-1, 1)

#for states_dict in states: 
#    plot_static_trajectories(states=states_dict, body_names=bodies[3], colors=bodies[4], scale=scale, centre_body=centre_body, xlim=xlim, ylim=ylim)

In [8]:
#anim = visualize_trajectories(states=states, body_names=bodies[3], colors=bodies[4], interval=20, scale=scale, centre_body=centre_body, xlim=xlim, ylim=ylim, save_path="mcrl.gif")
#HTML(anim.to_jshtml())